**Programmer:** python_scripts (Abhijith Warrier)

**PYTHON SCRIPT TO _UNDERSTAND HOW CPython MANAGES MEMORY USING REFERENCE COUNTING & CYCLIC GARBAGE COLLECTION_. 🧠🐍**

CPython uses a **dual system** for memory management:

1. **Reference Counting** (immediate cleanup)
2. **Cyclic Garbage Collector** (finds unreachable cycles)

In this DeepCut we explore:
- how reference counts work
- how to inspect them with `sys.getrefcount()`
- how cycles prevent refcount cleanup
- how the `gc` module finds and collects cycles
- how memory leaks occur in Python code

---

## 📦 Import Standard Library

In [1]:
import sys
import gc

---

## 🧩 Snippet 1 — CPython frees objects when their reference count hits zero

Every Python object tracks how many references point to it.
When the count drops to zero → memory is returned immediately.

In [2]:
x = [1, 2, 3]
print("Initial refcount:", sys.getrefcount(x))   # +1 from argument

y = x
print("After alias:", sys.getrefcount(x))

del y
print("After del y:", sys.getrefcount(x))

Initial refcount: 2
After alias: 3
After del y: 2


---

## 🔍 Snippet 2 — sys.getrefcount(obj) temporarily increments the count

The argument itself adds a temporary reference.

This explains the “+1 effect.”

In [3]:
a = []
print(sys.getrefcount(a))     # usually 2 (a + function arg)

2


---

## 🧠 Snippet 3 — Reference cycles cannot be freed immediately

If object A references B and B references A:

- Their reference counts never drop to zero
- CPython cannot free them via refcounting

These stay in memory until the cyclic GC steps in.

In [4]:
a = {}
b = {}

a["ref"] = b
b["ref"] = a

print("Refcount for a:", sys.getrefcount(a))
print("Refcount for b:", sys.getrefcount(b))

del a, b

# At this point, refcounts didn't hit zero due to mutual references

Refcount for a: 3
Refcount for b: 3


---

## 🔄 Snippet 4 — CPython’s cyclic GC finds unreachable cycles

The `gc` module tracks container objects and can detect cycles that reference counting misses.

In [5]:
gc.set_debug(gc.DEBUG_SAVEALL)

# Create a cycle
x = []
x.append(x)

del x

gc.collect()   # force collection

print("Unreachable objects:", gc.garbage)

Unreachable objects: [<bound method HelpFormatter._format_action of <IPython.core.magic_arguments.MagicHelpFormatter object at 0x108c60470>>, [_StoreAction(option_strings=['--proc'], dest='proc', nargs=None, const=None, default=None, type=<class 'str'>, choices=None, required=False, help='The variable in which to store Popen instance.\n            This is used only when --bg option is given.\n            ', metavar=None, deprecated=False)], (<bound method HelpFormatter._format_action of <IPython.core.magic_arguments.MagicHelpFormatter object at 0x108c60470>>, [_StoreAction(option_strings=['--proc'], dest='proc', nargs=None, const=None, default=None, type=<class 'str'>, choices=None, required=False, help='The variable in which to store Popen instance.\n            This is used only when --bg option is given.\n            ', metavar=None, deprecated=False)]), <bound method HelpFormatter._format_action of <IPython.core.magic_arguments.MagicHelpFormatter object at 0x108c60470>>, [_StoreTru

---

## ⚠️ Snippet 5 — A memory leak caused by lingering references

Reference cycles that include objects with __del__ can leak.
Functions storing global references can leak too.

In [6]:
LEAKS = []

class Node:
    def __init__(self):
        self.ref = self
    def __del__(self):
        pass  # destructor breaks automatic GC cycle handling

# create leak
n = Node()
LEAKS.append(n)
del n

gc.collect()
print("Leak count:", len(LEAKS))

Leak count: 1


---

## 🧹 Snippet 6 — Weak references prevent memory leaks in registries

Weak references let objects be collected even when referenced.

In [7]:
import weakref

registry = weakref.WeakValueDictionary()

class User:
    pass

u = User()
registry["user"] = u
print("Before GC:", dict(registry))

del u
gc.collect()

print("After GC:", dict(registry))   # empty → weakref avoided leak

Before GC: {'user': <__main__.User object at 0x108c6d2b0>}
After GC: {}


---

## ✅ One-liner Takeaway

**CPython frees memory using reference counting, but cycles require the cyclic GC — and careless references (or destructors) can still cause real memory leaks.**

---